In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/pga_tour_raw.csv")

In [4]:
df.shape

(36864, 37)

In [5]:
df.dtypes

Player_initial_last        str
tournament id            int64
player id                int64
hole_par                 int64
strokes                  int64
hole_DKP               float64
hole_FDP               float64
hole_SDP                 int64
streak_DKP               int64
streak_FDP             float64
streak_SDP               int64
n_rounds                 int64
made_cut                 int64
pos                    float64
finish_DKP               int64
finish_FDP               int64
finish_SDP               int64
total_DKP              float64
total_FDP              float64
total_SDP                int64
player                     str
Unnamed: 2             float64
Unnamed: 3             float64
Unnamed: 4             float64
tournament name            str
course                     str
date                       str
purse                  float64
season                   int64
no_cut                   int64
Finish                     str
sg_putt                float64
sg_arg  

In [7]:
df.head()

,Player_initial_last,tournament id,player id,hole_par,strokes,hole_DKP,hole_FDP,hole_SDP,streak_DKP,streak_FDP,...,purse,season,no_cut,Finish,sg_putt,sg_arg,sg_app,sg_ott,sg_t2g,sg_total
0,A. Ancer,401353224,9261,288,289,60.0,51.1,56,3,7.6,...,12.0,2022,0,T32,0.20,-0.13,-0.08,0.86,0.65,0.85
1,A. Hadwin,401353224,5548,288,286,72.5,61.5,61,8,13.0,...,12.0,2022,0,T18,0.36,0.75,0.31,0.18,1.24,1.60
2,A. Lahiri,401353224,4989,144,147,21.5,17.4,27,0,0.0,...,12.0,2022,0,CUT,-0.56,0.74,-1.09,0.37,0.02,-0.54
3,A. Long,401353224,6015,144,151,20.5,13.6,17,0,0.4,...,12.0,2022,0,CUT,-1.46,-1.86,-0.02,0.80,-1.08,-2.54
4,A. Noren,401353224,3832,144,148,23.5,18.1,23,0,1.2,...,12.0,2022,0,CUT,0.53,-0.36,-1.39,0.19,-1.56,-1.04


In [8]:
df.isnull().sum()

Player_initial_last        0
tournament id              0
player id                  0
hole_par                   0
strokes                    0
hole_DKP                   0
hole_FDP                   0
hole_SDP                   0
streak_DKP                 0
streak_FDP                 0
streak_SDP                 0
n_rounds                   0
made_cut                   0
pos                    15547
finish_DKP                 0
finish_FDP                 0
finish_SDP                 0
total_DKP                  0
total_FDP                  0
total_SDP                  0
player                     0
Unnamed: 2             36864
Unnamed: 3             36864
Unnamed: 4             36864
tournament name            0
course                     0
date                       0
purse                      0
season                     0
no_cut                     0
Finish                  7683
sg_putt                 7684
sg_arg                  7684
sg_app                  7684
sg_ott        

In [9]:
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])

In [10]:
df[['pos', 'Finish']].drop_duplicates().sort_values('pos').head(30)

,pos,Finish
16,1.0,1
31464,1.0,T13
1616,1.0,NaN
35564,1.0,T58
31506,2.0,T7
936,2.0,NaN
35597,2.0,T39
11,2.0,2
31451,2.0,T4
526,2.0,T2


In [11]:
df.groupby(['tournament id', 'player id']).size().sort_values(ascending=False).head(10)

tournament id  player id
2485           3950         2
2233           9131         2
2254           9131         2
3748           7081         2
2268           3688         2
2509           3950         2
401219796      7081         2
3781           10054        2
401353196      7081         2
2497           3950         2
dtype: int64

In [12]:
df[(df['tournament id'] == 2485) & (df['player id'] == 3950)]

,Player_initial_last,tournament id,player id,hole_par,strokes,hole_DKP,hole_FDP,hole_SDP,streak_DKP,streak_FDP,...,purse,season,no_cut,Finish,sg_putt,sg_arg,sg_app,sg_ott,sg_t2g,sg_total
31606,D. Lee,2485,3950,287,282,67.0,61.7,66,0,12.4,...,7.0,2016,0,T30,-1.21,0.73,0.01,0.49,1.23,0.03
31607,D. Lee,2485,3950,287,282,67.0,61.7,66,0,12.4,...,7.0,2016,0,CUT,2.43,0.16,-9.25,-1.28,-10.37,-7.94


In [13]:
# How many player-tournament pairs are affected?
dupe_counts = df.groupby(['tournament id', 'player id']).size()
dupes = dupe_counts[dupe_counts > 1]
print(len(dupes), "duplicate pairs out of", len(dupe_counts), "total")

# Does made_cut/no_cut help disambiguate which row is correct?
df[(df['tournament id'] == 2485) & (df['player id'] == 3950)][['made_cut', 'no_cut', 'Finish', 'sg_total']]

21 duplicate pairs out of 36843 total


,made_cut,no_cut,Finish,sg_total
31606,1,0,T30,0.03
31607,1,0,CUT,-7.94


In [14]:
dupe_counts = df.groupby(['tournament id', 'player id']).size()
dupes = dupe_counts[dupe_counts > 1]
print(len(dupes), "duplicate pairs out of", len(dupe_counts), "total")

21 duplicate pairs out of 36843 total


In [15]:
# Get the tournament/player pairs that have duplicates
dupe_counts = df.groupby(['tournament id', 'player id']).size()
dupe_keys = dupe_counts[dupe_counts > 1].index

# Build a boolean mask: True if this row's (tournament id, player id) is in the dupe set
is_dupe = df.set_index(['tournament id', 'player id']).index.isin(dupe_keys)

# Keep only rows NOT in the dupe set
df_clean = df[~is_dupe].reset_index(drop=True)

df_clean.shape

(36822, 34)